# Model-Derived Team Rankings

Uses EPA model residuals to compute team offensive efficiency:  
how much does each team's offense *beat expectations* given the situations they face?  

Also trains Success and Red Zone models, then produces composite rankings  
exported as JSON for the web page.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
import sys

pio.renderers.default = 'notebook_connected'
sys.path.insert(0, str(Path('../../').resolve()))

from python.features.engineering import add_features, get_feature_cols, time_split
from python.models.epa_model import EPAModel
from python.models.success_model import SuccessModel
from python.models.rz_model import RZModel, RZ_FEATURES
from python.pipeline.loader import load_all_metrics

DATA = Path('../../data/outputs/play_features.csv')

In [ ]:
# load models (must have run 01_epa_model.ipynb first)
epa_model     = EPAModel.load()
success_model = SuccessModel.load()
rz_model      = RZModel.load()
print('models loaded')

In [ ]:
df = pd.read_csv(DATA)
df_f = add_features(df)
feat = get_feature_cols()
feat = [c for c in feat if c in df_f.columns]

# predict on full dataset
df_f['pred_epa']     = epa_model.model.predict(df_f[feat])
df_f['epa_residual'] = df_f['epa'] - df_f['pred_epa']

feat_s = [c for c in success_model.feature_cols if c in df_f.columns]
df_f['pred_success'] = success_model.model.predict_proba(df_f[feat_s])[:, 1]

# RZ predictions on red zone subset
rz_mask = df_f['yardline_100'] <= 20
feat_rz = [c for c in rz_model.feature_cols if c in df_f.columns]
df_f.loc[rz_mask, 'pred_td_prob'] = rz_model.model.predict_proba(df_f.loc[rz_mask, feat_rz])[:, 1]
print('predictions generated')

In [ ]:
# offensive efficiency: EPA residual per team per season
off_eff = (
    df_f.groupby(['season', 'posteam'])
        .agg(
            plays=('epa', 'count'),
            mean_epa=('epa', 'mean'),
            mean_pred_epa=('pred_epa', 'mean'),
            epa_alpha=('epa_residual', 'mean'),   # over-performance vs expectation
            mean_success_prob=('pred_success', 'mean'),
        )
        .reset_index()
        .round(4)
)

print('2023 top offensive alphas (beats expectation most):')
print(off_eff[off_eff['season']==2023].sort_values('epa_alpha', ascending=False).head(10))

In [ ]:
# chart: EPA alpha by team (2023)
d23 = off_eff[off_eff['season']==2023].sort_values('epa_alpha', ascending=True)

fig = go.Figure(go.Bar(
    y=d23['posteam'],
    x=d23['epa_alpha'],
    orientation='h',
    marker_color=['#2271b3' if v >= 0 else '#c0392b' for v in d23['epa_alpha']]
))
fig.add_vline(x=0, line_dash='dash', line_color='black', line_width=1)
fig.update_layout(
    title='2023 Offensive EPA Alpha — Model Residual per Play',
    xaxis_title='EPA above/below model expectation',
    template='simple_white', height=700
)
fig.show()

In [ ]:
# composite ranking: actual EPA + model alpha + success rate + RZ
all_m = load_all_metrics()
epa_raw = all_m['epa'].rename(columns={'epa_per_play': 'raw_epa'})

rankings = (
    off_eff
    .merge(epa_raw[['season','posteam','raw_epa']], on=['season','posteam'], how='left')
    .merge(all_m['success_rate'][['season','posteam','success_rate']], on=['season','posteam'], how='left')
    .merge(all_m['redzone'][['season','posteam','rz_td_rate']], on=['season','posteam'], how='left')
    .merge(all_m['drive_efficiency'][['season','posteam','scoring_rate']], on=['season','posteam'], how='left')
)

for col in ['raw_epa','epa_alpha','success_rate','rz_td_rate','scoring_rate']:
    rankings[f'{col}_rank'] = rankings.groupby('season')[col].rank(ascending=False, method='min').astype(int)

rank_cols = [c for c in rankings.columns if c.endswith('_rank')]
rankings['composite'] = rankings[rank_cols].mean(axis=1).round(2)
rankings = rankings.sort_values(['season','composite'])

print('composite rankings (2023):')
print(rankings[rankings['season']==2023][['posteam','raw_epa','epa_alpha','success_rate','rz_td_rate','composite']].head(10))

In [ ]:
# export JSON for web
from python.export.json_exporter import export_metrics_json, export_prediction_grid

export_metrics_json()
export_prediction_grid(epa_model, success_model)
print('web data exported to web/data/')